# Fraud Detection — GPU Training (Google Colab)

Trains `transformer_xgboost` and `gnn` on a Colab T4 GPU and saves artifacts to Google Drive.

**Before running:**
1. Runtime → Change runtime type → **T4 GPU**
2. Run Cell 1 to verify GPU and mount Drive
3. Run Cell 2 to upload `train_transaction.csv` and `train_identity.csv` from your local `data/raw/` to Drive (one-time; future sessions reuse the Drive cache automatically)
4. Run all remaining cells in order

**Estimated time on T4:**
- `transformer_xgboost`: ~15–25 min (encoder 30 epochs + XGBoost)
- `gnn`: ~40–60 min (graph construction + 50 epochs + XGBoost)

**Output:** Trained artifacts saved to `MyDrive/fraud_detection/models/`

In [ ]:
# ── Cell 1: Verify GPU and mount Drive ────────────────────────────────────
import subprocess, sys

# Confirm T4 (or better) is assigned
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print('GPU:', result.stdout.strip() or 'NOT FOUND — go to Runtime > Change runtime type > T4 GPU')

import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

# Mount Google Drive for persistent artifact storage
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

In [ ]:
# ── Cell 2: Upload data to Google Drive (one-time) ────────────────────────
# Uploads train_transaction.csv and train_identity.csv from your local machine
# to MyDrive/fraud_detection/data/ so future sessions reuse them automatically.
#
# If you already ran this once and the files are in Drive, skip this cell.

import pathlib, shutil
from google.colab import files

DRIVE_DATA = pathlib.Path('/content/drive/MyDrive/fraud_detection/data')
DRIVE_DATA.mkdir(parents=True, exist_ok=True)

needed = [f for f in ['train_transaction.csv', 'train_identity.csv']
          if not (DRIVE_DATA / f).exists()]

if not needed:
    print('Both data files already in Drive — nothing to upload.')
else:
    print(f'Upload these files from your local data/raw/ directory:')
    for f in needed:
        print(f'  • {f}')
    print()
    uploaded = files.upload()   # triggers file picker — select both CSVs at once
    for fname, data in uploaded.items():
        dest = DRIVE_DATA / fname
        dest.write_bytes(data)
        print(f'  Saved {fname} to Drive ({len(data)/1e6:.0f} MB)')

print('Drive data directory:', list(DRIVE_DATA.iterdir()))

In [ ]:
# ── Cell 3: Clone repo and install dependencies ───────────────────────────
import subprocess, os, sys, pathlib

REPO_URL = 'https://github.com/jasmine2chen/IEEE-CIS-Fraud-Detection-Platform.git'
REPO_DIR = '/content/fraud_detection'

is_git_repo = (pathlib.Path(REPO_DIR) / '.git').exists()

if is_git_repo:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    print('Repo updated.')
else:
    # Remove stale directory (e.g. created by a previous data cell) before cloning
    if pathlib.Path(REPO_DIR).exists():
        import shutil
        shutil.rmtree(REPO_DIR)
        print(f'Removed stale {REPO_DIR}')

    r = subprocess.run(['git', 'clone', REPO_URL, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode == 0:
        print('Repo cloned from GitHub.')
    else:
        print(f'git clone failed:\n{r.stderr.strip()}')
        # Fallback: restore from a zip you uploaded to Drive
        DRIVE_CODE = pathlib.Path('/content/drive/MyDrive/fraud_detection/code.zip')
        if DRIVE_CODE.exists():
            os.makedirs(REPO_DIR, exist_ok=True)
            subprocess.run(['unzip', '-q', str(DRIVE_CODE), '-d', REPO_DIR], check=True)
            print(f'Repo restored from Drive zip.')
        else:
            raise RuntimeError(
                'Could not clone the repo and no Drive zip found.\n'
                'Options:\n'
                '  A) Update REPO_URL to your GitHub repo.\n'
                '  B) Zip your local fraud_detection/ folder (excluding data/raw, .venv, mlruns),\n'
                '     upload to MyDrive/fraud_detection/code.zip, then re-run this cell.'
            )

os.chdir(REPO_DIR)
assert os.getcwd() == REPO_DIR, f'chdir failed — got {os.getcwd()}'
assert (pathlib.Path(REPO_DIR) / 'src').exists(), f'src/ not found in {REPO_DIR} — clone incomplete'
print(f'Working directory: {os.getcwd()}')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.'], check=True)
print('Dependencies installed.')

In [ ]:
# ── Cell 4: Load data from Drive ──────────────────────────────────────────
import pathlib, shutil

# Always use absolute path — never relative, so cwd doesn't matter
data_dir = pathlib.Path(REPO_DIR) / 'data' / 'raw'
data_dir.mkdir(parents=True, exist_ok=True)

DRIVE_DATA = pathlib.Path('/content/drive/MyDrive/fraud_detection/data')

for fname in ['train_transaction.csv', 'train_identity.csv']:
    dst = data_dir / fname
    if dst.exists():
        print(f'  {fname} already in session ({dst.stat().st_size / 1e6:.0f} MB).')
        continue
    # Check both flat layout (Drive root) and raw/ subdirectory layout
    candidates = [DRIVE_DATA / fname, DRIVE_DATA / 'raw' / fname]
    src = next((p for p in candidates if p.exists()), None)
    if src:
        print(f'  Copying {fname} from {src} ...')
        shutil.copy(src, dst)
        print(f'  Done ({dst.stat().st_size / 1e6:.0f} MB)')
    else:
        raise FileNotFoundError(
            f'{fname} not found in Drive.\n'
            f'Checked: {[str(c) for c in candidates]}\n'
            'Run Cell 2 to upload your local data/raw/ files to Drive first.'
        )

print('\nData files ready:')
for f in sorted(data_dir.iterdir()):
    if f.suffix == '.csv':
        print(f'  {f}  ({f.stat().st_size / 1e6:.0f} MB)')

In [ ]:
# ── Cell 5: Copy pre-trained feature pipeline from Drive (if available) ───
# The feature pipeline is fitted on the training data and shared across models.
# If you trained XGBoost/MLP locally, upload the pipeline to Drive to reuse it.
# Otherwise, it will be rebuilt here (adds ~5 min).

import pathlib, shutil

DRIVE_MODELS = pathlib.Path('/content/drive/MyDrive/fraud_detection/models')
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)

local_pipeline = pathlib.Path('models/feature_pipeline.joblib')
local_pipeline.parent.mkdir(exist_ok=True)

drive_pipeline = DRIVE_MODELS / 'feature_pipeline.joblib'

if drive_pipeline.exists() and not local_pipeline.exists():
    shutil.copy(drive_pipeline, local_pipeline)
    print('Feature pipeline loaded from Drive.')
elif local_pipeline.exists():
    print('Feature pipeline already present locally.')
else:
    print('No pre-built pipeline found — will be built during training (~5 min).')
    print('TIP: After training, the pipeline is auto-saved to Drive for future sessions.')

In [ ]:
# ── Cell 6: Verify CUDA device selection ──────────────────────────────────
# train.py uses: cuda > mps > cpu
# On Colab T4, this will correctly pick CUDA.

import torch, time
device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Training device: {device}')
assert device == 'cuda', 'Expected CUDA — check Runtime > Change runtime type > T4 GPU'

# Quick sanity check: transformer attention forward pass timing on T4
x = torch.randn(1024, 256, 64, device=device)  # [B, seq_len, d_token]
# Warm up
for _ in range(3):
    q = x @ torch.randn(64, 64, device=device)
    k = x @ torch.randn(64, 64, device=device)
    _ = (q @ k.transpose(-2, -1)).softmax(-1)

t0 = time.perf_counter()
for _ in range(10):
    q = x @ torch.randn(64, 64, device=device)
    k = x @ torch.randn(64, 64, device=device)
    attn = (q @ k.transpose(-2, -1)).softmax(-1)
    _ = attn @ (x @ torch.randn(64, 64, device=device))
torch.cuda.synchronize()
ms = (time.perf_counter() - t0) / 10 * 1000
print(f'Attention fwd pass (1024×256×256): {ms:.1f} ms/batch')
print(f'Est. transformer encoder (30 epochs × 481 batches × {ms*3:.0f}ms fwd+bwd): {30*481*ms*3/60000:.1f} min')

In [ ]:
# ── Cell 7: Train TabTransformer → XGBoost ────────────────────────────────
import subprocess, time, pathlib, shutil, sys, os

REPO_DIR     = '/content/fraud_detection'
TRANS_CSV    = f'{REPO_DIR}/data/raw/train_transaction.csv'
ID_CSV       = f'{REPO_DIR}/data/raw/train_identity.csv'
DRIVE_MODELS = pathlib.Path('/content/drive/MyDrive/fraud_detection/models')
LOG_PATH     = '/content/transformer_train.log'

assert pathlib.Path(TRANS_CSV).exists(), f'Missing: {TRANS_CSV} — run Cell 4 first'

# Prepend REPO_DIR — don't replace existing PYTHONPATH (Colab sets /env/python)
_existing = os.environ.get('PYTHONPATH', '')
env = {**os.environ, 'PYTHONPATH': f'{REPO_DIR}:{_existing}' if _existing else REPO_DIR}

print('=' * 60)
print('Training TabTransformer → XGBoost')
print(f'Log → {LOG_PATH}')
print('=' * 60)

t0 = time.time()
with open(LOG_PATH, 'w') as log_f:
    result = subprocess.run(
        [sys.executable, '-m', 'src.train',
         '--trans', TRANS_CSV, '--id', ID_CSV, '--model', 'transformer_xgboost'],
        cwd=REPO_DIR, env=env,
        stdout=log_f, stderr=subprocess.STDOUT,
        text=True,
    )
elapsed = time.time() - t0

log_lines = open(LOG_PATH).readlines()
print(''.join(log_lines[-100:]))

status = 'FAILED' if result.returncode != 0 else 'OK'
print(f'\nTransformer training {status} in {elapsed/60:.1f} min  (exit code {result.returncode})')
if result.returncode == 0:
    drive_dst = DRIVE_MODELS / 'transformer_xgboost'
    drive_dst.mkdir(parents=True, exist_ok=True)
    for src in (pathlib.Path(REPO_DIR) / 'models' / 'transformer_xgboost').iterdir():
        shutil.copy(src, drive_dst / src.name)
    pipeline_src = pathlib.Path(REPO_DIR) / 'models' / 'feature_pipeline.joblib'
    if pipeline_src.exists():
        shutil.copy(pipeline_src, DRIVE_MODELS / 'feature_pipeline.joblib')
    print(f'Artifacts saved to Drive: {drive_dst}')

In [ ]:
# ── Cell 8: Train GraphSAGE → XGBoost ────────────────────────────────────
import subprocess, time, pathlib, shutil, sys, os

REPO_DIR     = '/content/fraud_detection'
TRANS_CSV    = f'{REPO_DIR}/data/raw/train_transaction.csv'
ID_CSV       = f'{REPO_DIR}/data/raw/train_identity.csv'
DRIVE_MODELS = pathlib.Path('/content/drive/MyDrive/fraud_detection/models')
LOG_PATH     = '/content/gnn_train.log'

assert pathlib.Path(TRANS_CSV).exists(), f'Missing: {TRANS_CSV} — run Cell 4 first'

_existing = os.environ.get('PYTHONPATH', '')
env = {**os.environ, 'PYTHONPATH': f'{REPO_DIR}:{_existing}' if _existing else REPO_DIR}

print('=' * 60)
print('Training GraphSAGE → XGBoost')
print('(graph construction is CPU-bound, may take 10–15 min before GPU starts)')
print(f'Log → {LOG_PATH}')
print('=' * 60)

t0 = time.time()
with open(LOG_PATH, 'w') as log_f:
    result = subprocess.run(
        [sys.executable, '-m', 'src.train',
         '--trans', TRANS_CSV, '--id', ID_CSV, '--model', 'gnn'],
        cwd=REPO_DIR, env=env,
        stdout=log_f, stderr=subprocess.STDOUT,
        text=True,
    )
elapsed = time.time() - t0

log_lines = open(LOG_PATH).readlines()
print(''.join(log_lines[-100:]))

status = 'FAILED' if result.returncode != 0 else 'OK'
print(f'\nGNN training {status} in {elapsed/60:.1f} min  (exit code {result.returncode})')
if result.returncode == 0:
    drive_dst = DRIVE_MODELS / 'gnn_xgboost'
    drive_dst.mkdir(parents=True, exist_ok=True)
    for src in (pathlib.Path(REPO_DIR) / 'models' / 'gnn_xgboost').iterdir():
        shutil.copy(src, drive_dst / src.name)
    print(f'Artifacts saved to Drive: {drive_dst}')

In [ ]:
# ── Cell 9: Run benchmark on all trained models ───────────────────────────
import pathlib, subprocess, sys, os

REPO_DIR  = '/content/fraud_detection'
TRANS_CSV = f'{REPO_DIR}/data/raw/train_transaction.csv'
ID_CSV    = f'{REPO_DIR}/data/raw/train_identity.csv'
env       = {**os.environ, 'PYTHONPATH': REPO_DIR}

model_artifacts = {
    'xgboost':             f'{REPO_DIR}/models/xgboost_fraud_model.joblib',
    'mlp_xgboost':         f'{REPO_DIR}/models/mlp_xgboost/xgboost.joblib',
    'transformer_xgboost': f'{REPO_DIR}/models/transformer_xgboost/xgboost.joblib',
    'gnn':                 f'{REPO_DIR}/models/gnn_xgboost/xgboost.joblib',
}

available = []
for name, path in model_artifacts.items():
    if pathlib.Path(path).exists():
        available.append(name)
        print(f'  ✓ {name}')
    else:
        print(f'  ✗ {name}  (not found — skipping)')

print(f'\nRunning benchmark on: {available}')
result = subprocess.run(
    [sys.executable, '-m', 'src.benchmark',
     '--trans', TRANS_CSV, '--id', ID_CSV,
     '--models', *available,
     '--output', f'{REPO_DIR}/reports/benchmark'],
    cwd=REPO_DIR, env=env, capture_output=False, text=True,
)
print(f'Benchmark exit code: {result.returncode}')

In [ ]:
# ── Cell 10: Download artifacts to local machine ──────────────────────────
# Zips trained models so you can download them from Colab → local.
# Alternative: they are already saved to Drive above.

import zipfile, pathlib, glob
from google.colab import files

ZIP_PATH = '/content/fraud_models_gpu.zip'

with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as zf:
    for pattern in [
        'models/transformer_xgboost/*',
        'models/gnn_xgboost/*',
        'models/feature_pipeline.joblib',
    ]:
        for f in glob.glob(pattern):
            zf.write(f)
            print(f'  added: {f}')

size_mb = pathlib.Path(ZIP_PATH).stat().st_size / 1e6
print(f'\nZip size: {size_mb:.1f} MB')
files.download(ZIP_PATH)

## After Downloading

Unzip into your local `fraud_detection/models/` directory:

```bash
cd ~/projects/fraud_detection
unzip ~/Downloads/fraud_models_gpu.zip -d .
```

Then run the full 4-model benchmark locally (inference is fast, no GPU needed):

```bash
python -m src.benchmark \
    --trans  data/raw/train_transaction.csv \
    --id     data/raw/train_identity.csv \
    --models xgboost mlp_xgboost transformer_xgboost gnn \
    --output reports/benchmark
```

The benchmark uses two-stage inference for hybrid models automatically — no manual steps needed.